# Model A: TF-IDF Logistic Regression (Constance Chen)

In [ ]:
import sys
from pathlib import Path

import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

import pandas as pd
from sklearn.base import clone

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import zipfile


In [ ]:
# locate shared project folder
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

sys.path.insert(0, str(project_dir))

# import data loading + splitting func
from data_utils import load_data, split_data

# kaggle zip
zip_path = project_dir / "data" / "cornell-tech-cs-5785-combined-2026-ai-vs-human-text.zip"

# load + split data
train, unlabeled, test = load_data(zip_path)
train_df, val_df = split_data(train)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)

## Baseline model: Word TF-IDF + Logistic Regression

In [ ]:
model_a = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True, # normalize words to lowercase
        ngram_range=(1, 2), # use individual words and consecutive word pairs
        min_df=2, # ignore terms appearing in fewer than 2 documents
        max_features=50000 # limit vocabulary to 50,000 features
    )),
    ("logreg", LogisticRegression(
        C=1.0, # regularization parameter
        max_iter=1000, # maximum optimization iterations
        random_state=42 # reproducibility
    ))
])

In [ ]:
# train model a using 8k training samples
start_time = time.perf_counter()

model_a.fit(train_df["text"], train_df["label"])

train_time = time.perf_counter() - start_time

print(f"Train time: {train_time:.2f} seconds")

## Model Evaluation

In [ ]:
# predict probabilities for class 1 (AI-generated) on validation set
train_probs = model_a.predict_proba(train_df["text"])[:, 1]
val_probs = model_a.predict_proba(val_df["text"])[:, 1]

# calc AUROC scores
train_auc = roc_auc_score(train_df["label"], train_probs)
val_auc = roc_auc_score(val_df["label"], val_probs)

print(f"Train AUROC: {train_auc:.4f}")
print(f"Validation AUROC: {val_auc:.4f}")

## Hyperparameter Tuning

In [ ]:
# test different regularization strengths
c_values = [0.01, 0.1, 1.0, 10.0, 100.0]
results = []

for c in c_values:
    # create fresh copy of baseline model
    model = clone(model_a)
    model.set_params(logreg__C=c)

    # train model and measure runtime
    start_time = time.perf_counter()
    model.fit(train_df["text"], train_df["label"])
    elapsed = time.perf_counter() - start_time

    # predict AI probabilities
    train_probs = model.predict_proba(train_df["text"])[:, 1]
    val_probs = model.predict_proba(val_df["text"])[:, 1]

    # calc AUROC
    train_auc = roc_auc_score(train_df["label"], train_probs)
    val_auc = roc_auc_score(val_df["label"], val_probs)

    results.append({
        "C": c,
        "Train AUROC": train_auc,
        "Val AUROC": val_auc,
        "Train time (s)": elapsed
    })

# compare results, ranked by validation AUROC
results_df = pd.DataFrame(results)
results_df.sort_values("Val AUROC", ascending=False)

### TF-IDF N-gram Tuning

In [ ]:
# test different word n-gram ranges
# keep C=10 and other tf-idf settings fixed
ngram_values = [(1, 1), (1, 2), (1, 3)]
ngram_results = []

for ngram in ngram_values:
    # create fresh copy of baseline model
    model = clone(model_a)

    # change n-gram range and use best C from prev experiment
    model.set_params(
        tfidf__ngram_range=ngram,
        logreg__C=10.0
    )

    # train model and measure runtime
    start_time = time.perf_counter()
    model.fit(train_df["text"], train_df["label"])
    elapsed = time.perf_counter() - start_time

    # predict AI probabilities
    train_probs = model.predict_proba(train_df["text"])[:, 1]
    val_probs = model.predict_proba(val_df["text"])[:, 1]

    # calc AUROC
    train_auc = roc_auc_score(train_df["label"], train_probs)
    val_auc = roc_auc_score(val_df["label"], val_probs)

    # store results for comparison
    ngram_results.append({
        "N-gram range": str(ngram),
        "Train AUROC": train_auc,
        "Val AUROC": val_auc,
        "Train time (s)": elapsed
    })

    print(f"Finished {ngram}: Val AUROC = {val_auc:.4f}")

# compare configurations, ranked by validation AUROC
ngram_df = pd.DataFrame(ngram_results)
ngram_df.sort_values("Val AUROC", ascending=False)


### TF-IDF min_df Tuning

In [ ]:
# test different minimum document frequencies
# keep C=10 and ngram_range=(1,3) fixed
min_df_values = [2, 3, 5]
min_df_results = []

for min_df in min_df_values:
    # create fresh copy of baseline model
    model = clone(model_a)

    # use best settings from prev experiments
    model.set_params(
        tfidf__ngram_range=(1, 3),
        tfidf__min_df=min_df,
        logreg__C=10.0
    )

    # train model and measure runtime
    start_time = time.perf_counter()
    model.fit(train_df["text"], train_df["label"])
    elapsed = time.perf_counter() - start_time

    # predict AI probabilities
    train_probs = model.predict_proba(train_df["text"])[:, 1]
    val_probs = model.predict_proba(val_df["text"])[:, 1]

    # calc AUROC
    train_auc = roc_auc_score(train_df["label"], train_probs)
    val_auc = roc_auc_score(val_df["label"], val_probs)

    # store results
    min_df_results.append({
        "min_df": min_df,
        "Train AUROC": train_auc,
        "Val AUROC": val_auc,
        "Train time (s)": elapsed
    })

    print(f"Finished min_df={min_df}: Val AUROC = {val_auc:.4f}")

# compare results by validation AUROC
min_df_df = pd.DataFrame(min_df_results)
min_df_df.sort_values("Val AUROC", ascending=False)


## Q2: Effect of Training Data Size
### Q2.1: Training and Validation AUROC

In [ ]:
# training sizes required by Q2.1
training_sizes = [500, 1000, 2000, 5000, 8000]
learning_curve_results = []

for n in training_sizes:
    # sample n examples from og training partition
    # stratify to maintain 50/50 human/AI label balance
    if n == len(train_df):
        subset = train_df.copy()
    else:
        subset, _ = train_test_split(
            train_df,
            train_size=n,
            stratify=train_df["label"],
            random_state=42
        )

    # create fresh model using best hyperparams
    model = clone(model_a)
    model.set_params(
        tfidf__ngram_range=(1, 3),
        tfidf__min_df=5,
        logreg__C=10.0
    )

    # train on only selected subset
    start_time = time.perf_counter()
    model.fit(subset["text"], subset["label"])
    elapsed = time.perf_counter() - start_time

    # calc training and validation AUROC
    train_probs = model.predict_proba(subset["text"])[:, 1]
    val_probs = model.predict_proba(val_df["text"])[:, 1]

    train_auc = roc_auc_score(subset["label"], train_probs)
    val_auc = roc_auc_score(val_df["label"], val_probs)

    # store results for report
    learning_curve_results.append({
        "Training rows": n,
        "Train AUROC": train_auc,
        "Validation AUROC": val_auc,
        "Train time (s)": elapsed
    })

    print(f"Finished {n} rows: Train AUROC = {train_auc:.4f}, "
          f"Val AUROC = {val_auc:.4f}")

# display
learning_curve_df = pd.DataFrame(learning_curve_results)
learning_curve_df.round(4)

### Q2.2: Learning Curve

In [ ]:
# plot training + validation AUROC vs training size
plt.figure(figsize=(8, 6))
plt.plot(learning_curve_df["Training rows"], learning_curve_df["Train AUROC"], marker='o', label='Train AUROC')
plt.plot(learning_curve_df["Training rows"], learning_curve_df["Validation AUROC"], marker='o', label='Validation AUROC')
plt.title('Learning Curve: AUROC vs Training Size')
plt.xlabel('Training Size (number of rows)')
plt.ylabel('AUROC')
plt.legend() 
plt.show()   

### Q2.3: Response in report template

## Kaggle Submission: Tuned logistic regression


Train the selected model on all 10,000 labeled examples and
generate predicted probabilities for the 3,000 test examples.

In [ ]:
# recreate best tuned LR model
final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 3),
        min_df=5,
        max_features=50000
    )),
    ("logreg", LogisticRegression(
        C=10.0,
        max_iter=1000,
        random_state=42
    ))
])

# train on all 10,000 labeled ex
final_model.fit(train["text"], train["label"])

# predict probability of AI-generated text for Kaggle test data
test_probs = final_model.predict_proba(test["text"])[:, 1]

print(f"Training examples: {len(train)}")
print(f"Test predictions: {len(test_probs)}")


In [ ]:
# load kaggle's sample submission to match the expected format
with zipfile.ZipFile(zip_path) as z:
    sample_file = next(
        name for name in z.namelist()
        if name.endswith("sample_submission.csv")
        and not name.startswith("__MACOSX/")
    )
    submission = pd.read_csv(z.open(sample_file))

# identify prediction column
prediction_cols = [col for col in submission.columns if col != "id"]
assert len(prediction_cols) == 1
prediction_col = prediction_cols[0]

# match predicted probabilities to correct test IDs
prob_by_id = pd.Series(test_probs, index=test["id"])
submission[prediction_col] = submission["id"].map(prob_by_id)

# verify submission before saving
assert len(submission) == len(test)
assert submission[prediction_col].notna().all()
assert submission[prediction_col].between(0, 1).all()

# save submission file locally
output_path = project_dir / "submission.csv"
submission.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(submission.head())
